In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [2]:
def display_missing_data_info(dataframe, ascending=False):
    missing_values_count = dataframe.isnull().sum()
    missing_values_percentage = (dataframe.isnull().mean() * 100)
    missing_data = pd.DataFrame({
        'Missing Values': missing_values_count,
        'Percentage (%)': missing_values_percentage
    })
    
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    sorted_missing_data = missing_data.sort_values(by='Missing Values', ascending=ascending)

    print(sorted_missing_data)
    return sorted_missing_data
    

In [3]:
installment = pd.read_csv('../../data/dseb63_installments_payments.csv')

In [4]:
installment['SK_ID_PREV'].nunique()

549020

In [5]:
installment.shape

(7744758, 8)

In [6]:
installment.head()

,SK_ID_PREV,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,SK_ID_CURR
0,1054186,1.0,6,-1180.0,-1187.0,6948.360,6948.360,147397.0
1,2452854,1.0,21,-546.0,-552.0,11302.605,11302.605,147397.0
2,1054186,1.0,2,-1300.0,-1307.0,6948.360,6948.360,147397.0
3,1682318,1.0,2,-240.0,-243.0,7374.510,7374.510,147397.0
4,2452854,1.0,10,-876.0,-882.0,11302.605,11302.605,147397.0


In [7]:
installment['INSTALLMENT_PAYMENT_DIFF'] = installment['AMT_INSTALMENT'] - installment['AMT_PAYMENT']

## Feature for installments payment
- Group by id prev 
- Count num install
- version: max
- Late: mean
- Early: mean
- Last payment date
- Total money
- mean

## Group by id curr
- Count install
- Sum money
- Mean early
- Mean late
- Mean value each payment
- Count last payment 3-6-9-12-24

### Sep by duration (less than 12 and more than 12) (ngan han / trung han)
- Count install 
- Sum money
- Mean early
- Mean late
- Mean value each payment
- Variance each payment
- Count payment 3-12-24
- Sum payment 3-12-24

In [8]:
installment['DIFF'] = installment['DAYS_INSTALMENT'] - installment['DAYS_ENTRY_PAYMENT']
# installment['DIFF'].fillna(0, inplace=True)
installment['LATE'] = installment['DIFF'] > 0
installment['EARLY'] = installment['DIFF'] <= 0

installment['LATE'] *= installment['DIFF']
installment['EARLY'] *= -installment['DIFF']

installment["LAST_30_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -30
installment["LAST_60_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -60
installment["LAST_90_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -90
installment["LAST_180_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -180
installment["LAST_365_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -365
installment["LAST_730_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -365*3 # 3 ye

installment["PAYMENT_LAST_30_DAYS"] = installment["LAST_30_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_60_DAYS"] = installment["LAST_60_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_90_DAYS"] = installment["LAST_90_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_180_DAYS"] = installment["LAST_180_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_365_DAYS"] = installment["LAST_365_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_730_DAYS"] = installment["LAST_730_DAYS"] * installment["AMT_PAYMENT"]



In [9]:
installment["LATENESS_LAST_90_DAYS"] = installment["LAST_90_DAYS"] * installment["LATE"]
installment["LATENESS_LAST_180_DAYS"] = installment["LAST_180_DAYS"] * installment["LATE"]
installment["LATENESS_LAST_365_DAYS"] = installment["LAST_365_DAYS"] * installment["LATE"]
installment["LATENESS_LAST_730_DAYS"] = installment["LAST_730_DAYS"] * installment["LATE"]

installment["EARLYNESS_LAST_90_DAYS"] = installment["LAST_90_DAYS"] * installment["EARLY"]
installment["EARLYNESS_LAST_180_DAYS"] = installment["LAST_180_DAYS"] * installment["EARLY"]
installment["EARLYNESS_LAST_365_DAYS"] = installment["LAST_365_DAYS"] * installment["EARLY"]
installment["EARLYNESS_LAST_730_DAYS"] = installment["LAST_730_DAYS"] * installment["EARLY"]

installment["PAYMENT_LATENESS_LAST_180_DAYS"] = installment["LAST_180_DAYS"] * installment["LATENESS_LAST_180_DAYS"]
installment["PAYMENT_LATENESS_LAST_365_DAYS"] = installment["LAST_365_DAYS"] * installment["LATENESS_LAST_365_DAYS"]
installment["PAYMENT_LATENESS_LAST_730_DAYS"] = installment["LAST_730_DAYS"] * installment["LATENESS_LAST_730_DAYS"]

installment["PAYMENT_EARLYNESS_LAST_180_DAYS"] = installment["LAST_180_DAYS"] * installment["EARLYNESS_LAST_180_DAYS"]
installment["PAYMENT_EARLYNESS_LAST_365_DAYS"] = installment["LAST_365_DAYS"] * installment["EARLYNESS_LAST_365_DAYS"]
installment["PAYMENT_EARLYNESS_LAST_730_DAYS"] = installment["LAST_730_DAYS"] * installment["EARLYNESS_LAST_730_DAYS"]

installment['INSTALLMENT_PAYMENT_DIFF_LAST_180_DAYS'] = installment['INSTALLMENT_PAYMENT_DIFF'] * installment['LAST_180_DAYS']
installment['INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS'] = installment['INSTALLMENT_PAYMENT_DIFF'] * installment['LAST_365_DAYS']
installment['INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS'] = installment['INSTALLMENT_PAYMENT_DIFF'] * installment['LAST_730_DAYS']


In [10]:
installment['COUNT'] = 1

In [11]:
gb_installment = installment.groupby('SK_ID_PREV').agg({'SK_ID_CURR':'first', 
                                                        'NUM_INSTALMENT_NUMBER':'max', 
                                                        'COUNT':'sum', 
                                                        'DIFF':['sum', 'mean', 'std'],
                                                        'LATE': ['sum', 'mean', 'std'], 
                                                        'EARLY': ['sum', 'mean', 'std'], 
                                                        'DAYS_ENTRY_PAYMENT':'max', 
                                                        'INSTALLMENT_PAYMENT_DIFF' : ['sum', 'mean', 'std'],
                                                        'AMT_PAYMENT':'sum',
                                                          
                                                        'LAST_30_DAYS':'sum', 
                                                        'LAST_60_DAYS':'sum', 
                                                        'LAST_90_DAYS':'sum', 
                                                        'LAST_180_DAYS':'sum', 
                                                        'LAST_365_DAYS':'sum', 
                                                        'LAST_730_DAYS':'sum', 
                                                        
                                                        'PAYMENT_LAST_30_DAYS':['sum', 'mean', 'std'],  # 'max',
                                                        'PAYMENT_LAST_60_DAYS':['sum', 'mean', 'std'], 
                                                        'PAYMENT_LAST_90_DAYS':['sum', 'mean', 'std', 'max','min'], 
                                                        'PAYMENT_LAST_180_DAYS':['sum', 'mean', 'std'], 
                                                        'PAYMENT_LAST_365_DAYS':['sum', 'mean', 'std', 'max','min'], 
                                                        'PAYMENT_LAST_730_DAYS':['sum', 'mean', 'std'],
                                                        
                                                        'LATENESS_LAST_90_DAYS':['sum', 'mean'],
                                                        'LATENESS_LAST_180_DAYS':['sum', 'mean', 'std'],
                                                        'LATENESS_LAST_365_DAYS':['sum', 'mean'],
                                                        'LATENESS_LAST_730_DAYS':['sum', 'mean', 'std'],
                                                        
                                                        'EARLYNESS_LAST_90_DAYS':['sum', 'mean'],
                                                        'EARLYNESS_LAST_180_DAYS':['sum', 'mean', 'std'],
                                                        'EARLYNESS_LAST_365_DAYS':['sum', 'mean'],
                                                        'EARLYNESS_LAST_730_DAYS':['sum', 'mean', 'std'],
                                                        
                                                        'PAYMENT_LATENESS_LAST_180_DAYS':['sum', 'mean'],
                                                        'PAYMENT_LATENESS_LAST_365_DAYS':['sum', 'mean', 'std', 'max'],
                                                        'PAYMENT_LATENESS_LAST_730_DAYS':['sum', 'mean', 'std'],
                                                        
                                                        'PAYMENT_EARLYNESS_LAST_180_DAYS':['sum', 'mean'],
                                                        'PAYMENT_EARLYNESS_LAST_365_DAYS':['sum', 'mean', 'std', 'max'],
                                                        'PAYMENT_EARLYNESS_LAST_730_DAYS':['sum', 'mean', 'std'],
                                                        
                                                        'INSTALLMENT_PAYMENT_DIFF_LAST_180_DAYS':['sum', 'mean', 'std'],
                                                        'INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS':['sum', 'mean', 'std', 'max'],
                                                        'INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS':['sum', 'mean', 'std', 'max'],
                                                        
                                                        }).reset_index()

gb_installment.columns = ['_'.join(col).strip() for col in gb_installment.columns.values]
# gb_installment.rename(columns={'COUNT':'NUM_INSTALMENT', 'LATE':'LATE_PAYMENT', 'EARLY':'EARLY_PAYMENT', 'DAYS_ENTRY_PAYMENT':'LASTEST_PAYMENT_DATE', 'AMT_PAYMENT':'TOTAL_PAYMENT'}, inplace=True)
gb_installment['MEAN_INSTALLMENT'] = gb_installment['AMT_PAYMENT_sum'] / gb_installment['NUM_INSTALMENT_NUMBER_max']

In [12]:
gb_installment.rename({ 'SK_ID_PREV_': 'SK_ID_PREV', 'SK_ID_CURR_first': 'SK_ID_CURR'}, axis=1, inplace=True)

In [13]:
gb_mean_installment = gb_installment.groupby('SK_ID_CURR').agg({'MEAN_INSTALLMENT':'mean'}).reset_index()

In [ ]:
gb_installment_by_curr = installment.groupby('SK_ID_CURR').agg({
                                                        'NUM_INSTALMENT_NUMBER':'max', 
                                                        'COUNT':'sum', 
                                                        'DIFF':['sum', 'mean', 'std'],
                                                        'LATE': ['sum', 'mean', 'std'], 
                                                        'EARLY': ['sum', 'mean', 'std'], 
                                                        'DAYS_ENTRY_PAYMENT':'max', 
                                                        'INSTALLMENT_PAYMENT_DIFF' : ['sum', 'mean', 'std'],
                                                        'AMT_PAYMENT':'sum',
                                                          
                                                        'LAST_60_DAYS':'sum', 
                                                        'LAST_90_DAYS':'sum', 
                                                        'LAST_180_DAYS':'sum', 
                                                        'LAST_365_DAYS':'sum', 
                                                        'LAST_730_DAYS':'sum', 
                                                        
                                                        'PAYMENT_LAST_30_DAYS':['sum', 'mean', 'std'], 
                                                        'PAYMENT_LAST_60_DAYS':['sum', 'mean', 'std'], 
                                                        'PAYMENT_LAST_90_DAYS':['sum', 'mean', 'std', 'max'], 
                                                        'PAYMENT_LAST_180_DAYS':['sum', 'mean', 'std'], 
                                                        'PAYMENT_LAST_365_DAYS':['sum', 'mean', 'std', 'max','min'], # drop min ?
                                                        'PAYMENT_LAST_730_DAYS':['sum', 'mean', 'std'],
                                                        
                                                        'LATENESS_LAST_90_DAYS':['sum', 'mean'],
                                                        'LATENESS_LAST_180_DAYS':['sum', 'mean', 'std'],
                                                        'LATENESS_LAST_365_DAYS':['sum', 'mean'],
                                                        'LATENESS_LAST_730_DAYS':['sum', 'mean', 'std', 'max'],
                                                        
                                                        'EARLYNESS_LAST_90_DAYS':['sum', 'mean'],
                                                        'EARLYNESS_LAST_180_DAYS':['sum', 'mean', 'std',],
                                                        'EARLYNESS_LAST_365_DAYS':['sum', 'mean'],
                                                        'EARLYNESS_LAST_730_DAYS':['sum', 'mean', 'std', 'max'],
                                                        
                                                        'PAYMENT_LATENESS_LAST_180_DAYS':['sum', 'mean'],
                                                        'PAYMENT_LATENESS_LAST_365_DAYS':['sum', 'mean', 'std'],
                                                        'PAYMENT_LATENESS_LAST_730_DAYS':['sum', 'mean', 'std', 'max'],
                                                        
                                                        'PAYMENT_EARLYNESS_LAST_180_DAYS':['sum', 'mean'],
                                                        'PAYMENT_EARLYNESS_LAST_365_DAYS':['sum', 'mean', 'std'],
                                                        'PAYMENT_EARLYNESS_LAST_730_DAYS':['sum', 'mean', 'std', 'max'],
                                                        
                                                        'INSTALLMENT_PAYMENT_DIFF_LAST_180_DAYS':['sum', 'mean', 'std'],
                                                        'INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS':['sum', 'mean', 'std', 'max'],
                                                        'INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS':['sum', 'mean', 'std','max'],
                                                        
                                                        }).reset_index()

gb_installment_by_curr.columns = ['_'.join(col).strip() for col in gb_installment_by_curr.columns.values]
gb_installment_by_curr.rename({ 'SK_ID_CURR_': 'SK_ID_CURR'}, axis=1, inplace=True)

In [15]:
gb_installment_short_term = gb_installment[gb_installment['NUM_INSTALMENT_NUMBER_max'] <= 12]
gb_installment_long_term = gb_installment[gb_installment['NUM_INSTALMENT_NUMBER_max'] > 12]

In [16]:
gb_installment_short_term

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_NUMBER_max,COUNT_sum,DIFF_sum,DIFF_mean,DIFF_std,LATE_sum,LATE_mean,LATE_std,...,INSTALLMENT_PAYMENT_DIFF_LAST_180_DAYS_std,INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS_sum,INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS_mean,INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS_std,INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS_max,INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS_sum,INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS_mean,INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS_std,INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS_max,MEAN_INSTALLMENT
0,1000001,117953.0,2,2,32.0,16.000000,14.142136,32.0,16.000000,14.142136,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34221.712500
1,1000003,6707.0,3,3,46.0,15.333333,1.527525,46.0,15.333333,1.527525,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4951.350000
2,1000004,187620.0,7,7,187.0,26.714286,17.046156,187.0,26.714286,17.046156,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4789.022143
3,1000005,83759.0,10,11,93.0,8.454545,10.289447,97.0,8.818182,9.897658,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14702.170500
4,1000007,25033.0,5,5,84.0,16.800000,5.674504,84.0,16.800000,5.674504,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11246.805000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549015,2843490,158969.0,4,4,-11.0,-2.750000,5.500000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4039.436250
549016,2843491,277329.0,10,10,142.0,14.200000,15.193566,142.0,14.200000,15.193566,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25421.985000
549017,2843492,36159.0,12,12,75.0,6.250000,4.330127,75.0,6.250000,4.330127,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21915.900000
549018,2843494,70155.0,2,2,20.0,10.000000,12.727922,20.0,10.000000,12.727922,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,491397.232500


In [17]:
gb_installment_short_term_by_curr = gb_installment_short_term.groupby('SK_ID_CURR').agg({'COUNT_sum':'sum', 
                                                                                         'NUM_INSTALMENT_NUMBER_max':'mean', 
                                                                                         'MEAN_INSTALLMENT':'mean', 
                                                                                         'LATE_sum':'mean', 
                                                                                         'EARLY_sum':'mean', 
                                                                                         'DAYS_ENTRY_PAYMENT_max':'max', 
                                                                                         'AMT_PAYMENT_sum':'sum', 
                                                                                         
                                                                                         'LAST_90_DAYS_sum':'sum', 
                                                                                         'LAST_180_DAYS_sum':'sum', 
                                                                                         'LAST_365_DAYS_sum':'sum', 
                                                                                         'LAST_730_DAYS_sum':'sum',  
                                                                                         
                                                                                         'PAYMENT_LAST_90_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_180_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_365_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_730_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_730_DAYS_sum':'sum', 
                                                                                         
                                                                                         }).reset_index()

# gb_installment_short_term_by_curr.columns = ['_'.join(col).strip() for col in gb_installment_short_term_by_curr.columns.values]
# gb_installment_short_term_by_curr.rename(columns={'SK_ID_CURR_': 'SK_ID_CURR'}, inplace=True)

gb_installment_short_term_by_curr.rename({ col: col + '_SHORT_TERM' for col in gb_installment_short_term_by_curr.columns if col != 'SK_ID_CURR' }, axis=1, inplace=True)

gb_installment_long_term_by_curr = gb_installment_long_term.groupby('SK_ID_CURR').agg({'COUNT_sum':'sum', 
                                                                                         'NUM_INSTALMENT_NUMBER_max':'mean', 
                                                                                         'MEAN_INSTALLMENT':'mean', 
                                                                                         'LATE_sum':'mean', 
                                                                                         'EARLY_sum':'mean', 
                                                                                         'DAYS_ENTRY_PAYMENT_max':'max', 
                                                                                         'AMT_PAYMENT_sum':'sum', 
                                                                                         
                                                                                         'LAST_90_DAYS_sum':'sum', 
                                                                                         'LAST_180_DAYS_sum':'sum', 
                                                                                         'LAST_365_DAYS_sum':'sum', 
                                                                                         'LAST_730_DAYS_sum':'sum',  
                                                                                         
                                                                                         'PAYMENT_LAST_90_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_180_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_365_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_730_DAYS_sum':'sum', 
                                                                                         'PAYMENT_LAST_730_DAYS_sum':'sum', 
                                                                                                                                                                                  
                                                                                         }).reset_index()

# gb_installment_long_term_by_curr.columns = ['_'.join(col).strip() for col in gb_installment_long_term_by_curr.columns.values]
# gb_installment_long_term_by_curr.rename(columns={'SK_ID_CURR_': 'SK_ID_CURR'}, inplace=True)

gb_installment_long_term_by_curr.rename({ col: col + '_LONG_TERM' for col in gb_installment_long_term_by_curr.columns if col != 'SK_ID_CURR' }, axis=1, inplace=True)

In [18]:
gb_installment_short_term_by_curr

,SK_ID_CURR,COUNT_sum_SHORT_TERM,NUM_INSTALMENT_NUMBER_max_SHORT_TERM,MEAN_INSTALLMENT_SHORT_TERM,LATE_sum_SHORT_TERM,EARLY_sum_SHORT_TERM,DAYS_ENTRY_PAYMENT_max_SHORT_TERM,AMT_PAYMENT_sum_SHORT_TERM,LAST_90_DAYS_sum_SHORT_TERM,LAST_180_DAYS_sum_SHORT_TERM,LAST_365_DAYS_sum_SHORT_TERM,LAST_730_DAYS_sum_SHORT_TERM,PAYMENT_LAST_90_DAYS_sum_SHORT_TERM,PAYMENT_LAST_180_DAYS_sum_SHORT_TERM,PAYMENT_LAST_365_DAYS_sum_SHORT_TERM,PAYMENT_LAST_730_DAYS_sum_SHORT_TERM
0,0.0,10,3.333333,21645.100000,104.333333,0.0,-50.0,208517.850,1,3,6,10,100287.765,122636.925,182678.850,208517.850
1,1.0,27,10.500000,33473.669318,78.000000,38.0,-378.0,725941.575,0,0,0,17,0.000,0.000,0.000,621150.075
2,3.0,12,3.666667,14482.491750,143.333333,0.0,-158.0,145581.885,0,1,6,12,0.000,43050.285,93841.740,145581.885
3,4.0,6,6.000000,8953.522500,103.000000,0.0,-2230.0,53721.135,0,0,0,0,0.000,0.000,0.000,0.000
4,5.0,10,10.000000,9818.644500,92.000000,0.0,-1808.0,98186.445,0,0,0,0,0.000,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167836,307501.0,33,8.250000,9476.366562,85.500000,0.0,-503.0,311748.975,0,0,0,10,0.000,0.000,0.000,76119.660
167837,307503.0,10,10.000000,4494.393000,113.000000,0.0,-2013.0,44943.930,0,0,0,0,0.000,0.000,0.000,0.000
167838,307504.0,5,8.000000,3073.449375,63.500000,1.5,-46.0,26651.745,2,4,4,4,11777.760,23555.520,23555.520,23555.520
167839,307506.0,42,10.500000,11162.210625,89.750000,12.5,-66.0,518373.360,2,4,11,36,27530.145,55067.265,151447.185,500960.610


In [19]:
gb_installment_by_curr = gb_installment_by_curr.merge(gb_installment_short_term_by_curr, on='SK_ID_CURR', how='left')
gb_installment_by_curr = gb_installment_by_curr.merge(gb_installment_long_term_by_curr, on='SK_ID_CURR', how='left')
gb_installment_by_curr

,SK_ID_CURR,NUM_INSTALMENT_NUMBER_max,COUNT_sum,DIFF_sum,DIFF_mean,DIFF_std,DIFF_max,LATE_sum,LATE_mean,LATE_std,...,DAYS_ENTRY_PAYMENT_max_LONG_TERM,AMT_PAYMENT_sum_LONG_TERM,LAST_90_DAYS_sum_LONG_TERM,LAST_180_DAYS_sum_LONG_TERM,LAST_365_DAYS_sum_LONG_TERM,LAST_730_DAYS_sum_LONG_TERM,PAYMENT_LAST_90_DAYS_sum_LONG_TERM,PAYMENT_LAST_180_DAYS_sum_LONG_TERM,PAYMENT_LAST_365_DAYS_sum_LONG_TERM,PAYMENT_LAST_730_DAYS_sum_LONG_TERM
0,0.0,21,30,327.0,10.900000,19.353962,64.0,327.0,10.900000,19.353962,...,-5.0,102424.545,13.0,17.0,20.0,20.0,93047.265,94653.045,102424.545,102424.545
1,1.0,13,40,77.0,1.925000,11.425090,30.0,160.0,4.000000,8.661734,...,-18.0,61425.000,6.0,11.0,13.0,13.0,33750.000,56925.000,61425.000,61425.000
2,3.0,31,43,498.0,11.581395,21.434543,99.0,498.0,11.581395,21.434543,...,-5.0,107919.720,9.0,17.0,31.0,31.0,23897.160,47760.435,107919.720,107919.720
3,4.0,6,6,103.0,17.166667,3.188521,23.0,103.0,17.166667,3.188521,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,10,10,92.0,9.200000,3.614784,17.0,92.0,9.200000,3.614784,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180728,307501.0,12,33,342.0,10.363636,7.936538,32.0,342.0,10.363636,7.936538,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180729,307503.0,10,10,113.0,11.300000,2.263233,15.0,113.0,11.300000,2.263233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180730,307504.0,123,133,414.0,3.112782,13.057364,36.0,674.0,5.067669,9.552245,...,-22.0,191707.560,3.0,6.0,12.0,36.0,356.805,713.610,1427.220,4281.660
180731,307506.0,12,42,309.0,7.357143,7.244246,18.0,359.0,8.547619,4.209179,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
del gb_installment, gb_installment_short_term, gb_installment_long_term, gb_installment_short_term_by_curr, gb_installment_long_term_by_curr

In [21]:
gb_installment_by_curr.to_parquet('../../data/dseb63_installment_gb_v1_2.parquet', index=False)